In [7]:
import numpy as np
import scipy.stats

from pygom import SimulateOde, Transition, TransitionType
from pygom.utilR import rgamma
from pygom.model import common_models

n_sim = 1000
# initial time
t0 = 0
# the initial state, normalized to zero one
x0 = [1, 1.27e-6, 0]
# set the time sequence that we would like to observe
t = np.linspace(0, 150, 100)
# Standard.  Find the solution.
ode = common_models.SIR_norm()
ode.parameters = [0.5, 1.0 / 3.0]
ode.initial_values = (x0, t0)
solution = ode.integrate(t[1::], full_output=False)


In [9]:
ode._parameter_store.all_values_set

True

In [2]:
ode.ode._getEvalParam(state=x0, time=t[99])

[1, 1.27e-06, 0, np.float64(150.0), 0.5, 0.3333333333333333]

In [3]:

# now we need to define our ode explicitly
state_list = ['S', 'I', 'R']
param_list = ['beta', 'gamma']
transition_list = [
    Transition(origin='S', destination='I',
                equation='beta*S*I',
                transition_type=TransitionType.T),
    Transition(origin='I', destination='R',
                equation='gamma*I',
                transition_type=TransitionType.T)
]
# our stochastic version
odeS = SimulateOde(state_list, param_list,
                        transition=transition_list)

In [ ]:
d = dict()
d['beta'] = scipy.stats.gamma(100.0, 0.0, 1.0/200.0)
d['gamma'] = scipy.stats.gamma(100.0, 0.0, 1.0/300.0)
odeS.parameters = d
odeS.initial_values = (x0, t0)

# now we generate the solutions
sim = odeS.solve_determ(t[1::], n_sim, parallel=False)
solution_diff = sim - solution

# test :)
np.any(abs(solution_diff) <= 0.2)

np.True_

In [5]:
odeS.ode._getEvalParam(x0, t0)

[1,
 1.27e-06,
 0,
 0,
 np.float64(0.44927085061496774),
 np.float64(0.3347208644656678)]

In [6]:
test = d['beta']

test.rvs()

np.float64(0.5219026562854685)